# Dynamic Pricing for Perishable Inventory — Pipeline Notebook

This notebook calls the modular pipeline (`config.py`, `data_loading.py`, `demand_forecast.py`, etc.) stage by stage so you can see intermediate results.

**Before running:** activate the conda env (`conda activate dynamic_pricing`) and start Jupyter from the `code/` folder so the imports resolve.

To run everything in one shot: `Cell` → `Run All`.

To run one stage at a time: click into a cell and `Shift+Enter`.


## 0 · Setup

Make sure the current working directory is `code/` so the modules can be imported.

In [ ]:
import os, sys
from pathlib import Path

# Auto-detect: if we're not in code/, try to fix it
cwd = Path(os.getcwd())
if cwd.name != "code":
    candidate = cwd / "code"
    if candidate.exists():
        os.chdir(candidate)
print(f"Working directory: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)


In [ ]:
import config, data_loading, features, demand_forecast, elasticity, spoilage, optimizer, simulation, sensitivity

config.ensure_dirs()
print("All modules imported.")
print("Outputs will be written to:", config.OUTPUTS_DIR)
print("Models will be written to :", config.MODELS_DIR)


## 1 · Load perishable dataset & build the 3-way split

- **Train** : everything before the last 90 days  
- **Validation** : days -90 to -61 (used for model selection only)  
- **Holdout** : last 60 days (sacred — not touched until Stage 4)

In [ ]:
perishable = data_loading.load_perishable()
print(f"Cleaned rows: {len(perishable):,}  |  Categories: {sorted(perishable['category'].unique())}")
perishable.head()


In [ ]:
ts_pivot, price_daily = data_loading.build_daily_aggregates(perishable)
train_dates, val_dates, holdout_dates = data_loading.make_three_way_split(ts_pivot)
print(f"Train  : {train_dates[0].date()} → {train_dates[-1].date()}  ({len(train_dates)} days)")
print(f"Val    : {val_dates[0].date()} → {val_dates[-1].date()}  ({len(val_dates)} days)")
print(f"Holdout: {holdout_dates[0].date()} → {holdout_dates[-1].date()}  ({len(holdout_dates)} days)  ← SACRED")


## 2 · Demand forecasting (Stage 1)

Trains LightGBM / Prophet / Holt-Winters on the training window, evaluates on validation, picks the winner by directional accuracy.

**Tip:** Prophet is the slowest. The `RUN_PROPHET = False` flag below skips it for the first run. LightGBM still wins regardless — re-run later with `RUN_PROPHET = True` to get the full comparison table.

In [ ]:
# Toggle Prophet on/off (False = much faster first run)
RUN_PROPHET = False

feats = features.build_demand_feature_set(
    ts_pivot, price_daily, train_dates, val_dates, holdout_dates,
)
print(f"Train rows: {len(feats['feat_train'])}, Val rows: {len(feats['feat_val'])}, Holdout rows: {len(feats['feat_holdout'])}")


In [ ]:
print("[LightGBM] training + 5-fold TS-CV + validation eval...")
lgb_res = demand_forecast.evaluate_lightgbm(
    feats["X_train"], feats["y_train"], feats["X_val"], feats["y_val"], feats["feat_val"],
)
print(f"LightGBM val_RMSE={lgb_res['val_rmse']:.2f}  val_MAE={lgb_res['val_mae']:.2f}  dir_acc={lgb_res['dir_acc']:.2f}%")

print("\nLightGBM 5-fold CV results:")
display(lgb_res["cv_results"])


In [ ]:
ts_train = ts_pivot.loc[train_dates]

if RUN_PROPHET:
    print("[Prophet] per-category fits (this is the slow one — 2-5 min)...")
    pro_res = demand_forecast.evaluate_prophet(ts_train, val_dates, holdout_dates, feats["feat_val"])
    print(f"Prophet val_RMSE={pro_res['val_rmse']:.2f}  dir_acc={pro_res['dir_acc']:.2f}%")
else:
    pro_res = {"name": "Prophet", "val_rmse": 9e9, "val_mae": 9e9, "dir_acc": 0.0,
               "cat_rmse": pd.Series(dtype=float)}
    print("[Prophet] SKIPPED (set RUN_PROPHET=True above to include it)")


In [ ]:
print("[Holt-Winters] per-category fits...")
hw_res = demand_forecast.evaluate_holtwinters(ts_train, val_dates, feats["feat_val"])
print(f"Holt-Winters val_RMSE={hw_res['val_rmse']:.2f}  dir_acc={hw_res['dir_acc']:.2f}%")


In [ ]:
winner_name, comparison_df = demand_forecast.select_winner(lgb_res, pro_res, hw_res)
print(f"\n>>> Winner: {winner_name}")

display(comparison_df)
comparison_df.to_csv(config.OUTPUTS_DIR / "demand_model_comparison.csv", index=False)
print("Saved → outputs/demand_model_comparison.csv")


In [ ]:
print(f"[{winner_name}] retraining on train+val and forecasting holdout...")
cat_forecasts = demand_forecast.forecast_holdout(
    winner_name, ts_pivot, feats["feat_train"], feats["feat_val"], feats["feat_holdout"], holdout_dates,
)
print(f"category-level forecasts produced: {len(cat_forecasts)} entries")


## 3 · Elasticity OOS validation (Stage 2)

Estimates department-level log-log elasticity from Dunnhumby with store + week fixed effects, then validates on weeks 81–102.

Set `SKIP_DUNN = True` to use the precomputed elasticities in `config.DEFAULT_ELASTICITY_MAP` (saves ~2 min).

In [ ]:
SKIP_DUNN = False

if SKIP_DUNN or not config.DUNN_DIR.exists():
    print("[skipping Dunnhumby — using DEFAULT_ELASTICITY_MAP]")
    elasticity_map = dict(config.DEFAULT_ELASTICITY_MAP)
    oos_df = pd.DataFrame([
        {"Department": d, "Elasticity": v, "N_Train": np.nan, "N_Test": np.nan,
         "InSample_R2": np.nan, "OOS_R2": np.nan}
        for d, v in elasticity_map.items()
    ])
else:
    print("Loading Dunnhumby transactions + product table...")
    elas_df = elasticity.load_dunnhumby_elasticity_table()
    print(f"Dunnhumby aggregated rows: {len(elas_df):,}")
    oos_df = elasticity.run_elasticity_validation(elas_df)
    elasticity_map = elasticity.elasticity_map_from_validation(oos_df)

display(oos_df)
oos_df.to_csv(config.OUTPUTS_DIR / "elasticity_estimates.csv", index=False)
print("Saved → outputs/elasticity_estimates.csv")


## 4 · Spoilage classifier (Stage 3)

Trains LogisticRegression / RandomForest / XGBoost / LightGBM on the binary target (`units_wasted > 0`) using only pre-holdout rows. Picks winner by ROC-AUC, with Brier as tiebreaker.

In [ ]:
df_opt, df_pre, df_hold = spoilage.prepare_spoilage_dataset(perishable, holdout_dates)
print(f"Pre-holdout rows: {len(df_pre):,}  |  Holdout rows: {len(df_hold):,}")


In [ ]:
print("Training 4 spoilage classifiers (this takes 1-2 min)...")
spoil_model, train_cols, spoil_cmp_df, spoil_winner = spoilage.train_and_select(df_pre)

display(spoil_cmp_df)
spoil_cmp_df.to_csv(config.OUTPUTS_DIR / "spoilage_model_comparison.csv", index=False)
print(f"\n>>> Winner: {spoil_winner}")


In [ ]:
df_hold, df_opt = spoilage.score_holdout(spoil_model, train_cols, df_hold, df_opt)
waste_frac = spoilage.empirical_waste_fraction(df_pre)
print(f"Empirical waste fraction (pre-holdout): {waste_frac:.4f}")
print(f"Holdout spoilage_risk — mean: {df_hold['spoilage_risk'].mean():.3f}  max: {df_hold['spoilage_risk'].max():.3f}")


In [ ]:
# Persist the trained spoilage model for the dashboard
import pickle
with open(config.MODELS_DIR / "spoilage_model.pkl", "wb") as f:
    pickle.dump({"model": spoil_model, "train_cols": train_cols,
                 "winner": spoil_winner, "waste_frac": waste_frac}, f)
print(f"Saved → models/spoilage_model.pkl")


## 5 · 60-day OOS simulation (Stage 4)

Scales category-level demand forecasts to per-SKU level, then runs all three pricing strategies on the holdout window.

In [ ]:
sku_forecasts = optimizer.scale_forecasts_to_sku(cat_forecasts, df_hold)
sku_vals = list(sku_forecasts.values())
print(f"Scaled forecasts: {len(sku_forecasts)} (date, category) entries")
print(f"  Mean SKU-day forecast: {np.mean(sku_vals):.1f}")
print(f"  Actual holdout units_sold mean: {df_hold['units_sold'].mean():.1f}")
print("(these two should be in the same ballpark — that confirms the scaling is calibrated)")


In [ ]:
sim_df = df_hold.copy().reset_index(drop=True)
sim_kwargs = dict(spoilage_model=spoil_model, train_cols=train_cols,
                  elasticity_map=elasticity_map, waste_frac=waste_frac,
                  demand_forecasts=sku_forecasts)

print("Running 60-day simulation across 3 strategies (1-2 min)...")
sim_results = simulation.run_full_simulation(sim_df, **sim_kwargs)
sim_results.to_csv(config.OUTPUTS_DIR / "sim_results.csv", index=False)
print(f"Simulation rows: {len(sim_results):,}  →  saved to outputs/sim_results.csv")


In [ ]:
kpi = simulation.kpi_summary(sim_results)
kpi.to_csv(config.OUTPUTS_DIR / "strategy_kpi.csv")
display(kpi)


In [ ]:
delta = simulation.delta_table(kpi, winner_name, spoil_winner)
delta.to_csv(config.OUTPUTS_DIR / "strategy_delta_table.csv", index=False)
display(delta)

base_profit = kpi.loc["no_discount", "Total_Profit"]
dyn_profit = kpi.loc["dynamic", "Total_Profit"]
print(f"\nDynamic delivers ${dyn_profit:,.0f} profit vs ${base_profit:,.0f} baseline")
print(f"= {dyn_profit / base_profit:.2f}x improvement" if base_profit > 0 else "= no profitable baseline to compare against")


In [ ]:
cat_breakdown = simulation.category_profit_breakdown(sim_results)
cat_breakdown.to_csv(config.OUTPUTS_DIR / "category_profit.csv", index=False)
display(cat_breakdown)


In [ ]:
bucket_tbl = simulation.near_expiry_buckets(sim_results)
bucket_tbl.to_csv(config.OUTPUTS_DIR / "near_expiry_buckets.csv", index=False)
print("Optimizer behaviour by days-until-expiry (Dynamic strategy):")
display(bucket_tbl)


In [ ]:
daily_tbl = simulation.daily_profit_trend(sim_results)
daily_tbl.to_csv(config.OUTPUTS_DIR / "daily_profit_trend.csv", index=False)
daily_tbl.head()


## 6 · Sensitivity analyses (Stage 5)

Three checks:
1. **High-spoilage subset** — re-run all 3 strategies on items with `days_until_expiry ≤ 3`.
2. **Elasticity ±50%** — both on the full sample AND on non-near-expiry rows only (the one that actually tests whether the regular elasticity is doing work).
3. **Optimizer stability** — discount distribution on a 1k-row pre-holdout sample.

In [ ]:
print("Running high-spoilage scenario...")
hs_kpi, hs_n = sensitivity.high_spoilage_kpi(sim_df, **sim_kwargs)
hs_kpi.to_csv(config.OUTPUTS_DIR / "high_spoilage_kpi.csv")
print(f"High-spoilage subset: {hs_n} rows")
display(hs_kpi)


In [ ]:
print("Running elasticity sensitivity (this re-runs the optimizer 6 times)...")
elas_sens = sensitivity.elasticity_sensitivity(
    sim_df, elasticity_map=elasticity_map,
    **{k: v for k, v in sim_kwargs.items() if k != "elasticity_map"})
elas_sens.to_csv(config.OUTPUTS_DIR / "elasticity_sensitivity.csv", index=False)
display(elas_sens)


In [ ]:
print("Running optimizer stability check on 1k pre-holdout sample (1-2 min)...")
opt_df = sensitivity.optimizer_stability_sample(
    df_pre, spoilage_model=spoil_model, train_cols=train_cols,
    elasticity_map=elasticity_map, waste_frac=waste_frac, sample_n=1000,
)
opt_df.to_csv(config.OUTPUTS_DIR / "optimizer_stability_sample.csv", index=False)

dist_df = sensitivity.discount_distribution(opt_df)
dist_df.to_csv(config.OUTPUTS_DIR / "discount_distribution.csv", index=False)
print("Discount distribution (% of SKUs):")
display(dist_df)


In [ ]:
cat_disc_df = sensitivity.category_mean_discount(opt_df)
cat_disc_df.to_csv(config.OUTPUTS_DIR / "category_mean_discount.csv", index=False)
print("Mean recommended discount by category:")
display(cat_disc_df)


## 7 · Write manifest and confirm done

In [ ]:
import json
manifest = {
    "run_completed_at": pd.Timestamp.now().isoformat(),
    "winner_demand_model": winner_name,
    "winner_spoilage_model": spoil_winner,
    "elasticity_map": elasticity_map,
    "empirical_waste_fraction": waste_frac,
    "n_holdout_dates": len(holdout_dates),
    "n_holdout_rows": len(df_hold),
    "ran_prophet": RUN_PROPHET,
    "ran_dunn": not SKIP_DUNN,
    "n_categories": int(perishable["category"].nunique()),
}
(config.OUTPUTS_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest, indent=2, default=str))


In [ ]:
# List all output files so you can confirm everything is in place
print("Files in outputs/:")
for f in sorted(config.OUTPUTS_DIR.glob("*")):
    print(f"  {f.name:40s} {f.stat().st_size:>10,} bytes")
print("\nFiles in models/:")
for f in sorted(config.MODELS_DIR.glob("*")):
    print(f"  {f.name:40s} {f.stat().st_size:>10,} bytes")


---
## Done

If every cell ran without errors and the KPI table shows Dynamic strategy beating the baseline, you're ready for the dashboard step.

Send me:
- A screenshot of the KPI table (or paste `outputs/strategy_kpi.csv`)
- The `outputs/manifest.json` content
- Any cell that errored

Once verified, I'll build the Streamlit dashboard that reads from these CSVs.
